In [0]:
df = spark.read.table("teams.sensorx.df_sx_800")

display(df.limit(10))teams.sensorx.df_sx_800

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col


# Prepare features and label columns
feature_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "payload_xrayController_onTime"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_features = assembler.transform(df).withColumn("label", col("payload_fault_state").cast("double"))

# Split the data into 80% training and 20% test
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(maxIter=10, regParam=0.3, elasticNetParam=0.8)

# Fit the model on the 80% training data
lrModel = lr.fit(train_df)

# Print the coefficients and intercept for logistic regression
print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))